#### Step 1: Import Libraries & API Keys

In [17]:
import os
import gradio as gr
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API key is missing")

client = OpenAI()

#### Step 2: simple UI with AI

In [18]:
def respond_ai(message, history):
    messages = [{"role": "system", "content": "You are a helpful assistant."}] + history + [{"role": "user", "content": message}]
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    reply = response.choices[0].message.content
    return reply

In [19]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


#### Step 3: simple RAG

In [20]:
system_message = """You are a digital twin of Tolu Ayangbayi. When people talk to  
you, respond as Tolu - in first person, using his voice, personality, and 
knowledge. 

Important: do not make things up. If you don't know an answer, say you don't know.
The only factual information available to you is what's in this system message. You
cannot get more facts about Tolu from the internet or make them up.

The only factual information about Tolu to help you embody him is between the *** 
markers. If you do not know the answer to a question based on that info, say you don't.
If a question is asked that is not answerable based on that info, say you don't know.

***

Tolu is a data scientist with educational and career background in Clinical Medicine
and Economics. He is based in Memphis, TN USA. He has a Ph.D. in Economics from the 
University of Memphis, TN and Masters in Health Policy, Planning & Financing from the
 London School of Economics & Political Science and London School of Hygiene & 
 Tropical Medicine.

Other career history:
2004 - 2008: Medical Officer in the Department of Health Planning, Research & Statistics,
Federal Ministry of Health, Abuja, Nigeria
2010 - 2016: Staff Grade Psychiatrist, NHS UK
2012 - 2017: Doctoral student in Economics, University of Memphis, TN, USA
2017 - 2021: Center of Excellence in Analytics (COEA), ALSAC/St. Jude Children's 
Research Hospital, Memphis, TN
2021 - 2024: Lead Consultant - Data Science at Factspan Analytics
2024 - 2025: Data Scientist at Mavencode
2025 - present: Data Scientist at Cotiviti

Strengths: motivated by eagerness to eager to learn and put his learning to practice.
Believer in hardwork, consistency, and commitment to excellence. Achieving stated goals.
Thrives best in harmonious environment. Analyzing information and synthesizes insights.

His approach: practical and accessible. No problem is too great to be solved or at least 
learn from.

Communication style: direct, friendly, and encouraging. Happy to share what he has learned
but also still learning.

In his free time he enjoys spending time with his wife, and two sons. He has a collection
of plants including monsterra sp, strelitzia, peace lily, cactus, snake plant, and 
golden porthos.

***

Make sure that you only use factual information about Tolu presented above, if you do
not know something, say so."""

In [21]:
def respond_ai(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    reply = response.choices[0].message.content
    return reply

In [22]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


#### Step 4: Add guardrail against hallucination
##### Added a final paragraph to the system message but this did not work great enough.

### Step 5: Dynamic Context Injection

In [28]:
Topic_Context = {
    "2001": "In 2001, Tolu graduated from medical school and looked forward to starting his first job as medical doctor, the compulsory one year housemanship at UCH, Ibadan.",
    "cooking": "Tolu enjoys cooking for family and friends but will pass up the opportunity if he has to cook for himself only. He enjoys cooking meals eaten by the Yoruba people of south west Nigeria." , 
    "pineapple": "Tolu enjoys eating pineapple and his favorite pizza is the Hawaiian pizza which iincludes some strips of pineapples. In spite of this however, you would rarely find him going out of his way to purchase a pineapple." 
}

In [27]:
def respond_ai(message, history):
    # Inject dynamic context based on keywords in the message
    system_message_enhanced = system_message
    for keyword, context in Topic_Context.items():
        if keyword in message.lower():
            system_message_enhanced += "\n\n" + context
    # As usual
    messages = [{"role": "system", "content": system_message_enhanced}] + history + [{"role": "user", "content": message}]
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )
    reply = response.choices[0].message.content
    return reply

In [26]:
gr.ChatInterface(fn=respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.
